In [ ]:
import tkinter as tk
import pygame
import os
from tkinter import messagebox
from typing import List
# Инициализация Pygame
pygame.init()


WIDTH = 400
HEIGHT = 400  # Размеры окна


WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
RED = (255, 0, 0)
BLUE = (0, 0, 255)  # Цвета
GREEN =()
YELLOW =()

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Шахматная доска")
clock = pygame.time.Clock()


N = 0
L = 0
K = 0

root = tk.Tk()
root.title("Шахматные фигуры")
root.geometry('400x200')
root.resizable(True, True)
# Создание полей ввода
n, n_entry = tk.Label(root, text="Размер доски (N):"), tk.Entry(width=20)
n.pack()
n_entry.pack()
n_entry.focus()

l, l_entry = tk.Label(root, text="Количество фигур (L):"), tk.Entry(width=20)
l.pack()
l_entry.pack()

k, k_entry = tk.Label(root, text="Количество стоящих фигур (K):"), tk.Entry(width=20)
k.pack()
k_entry.pack()

#entry_K = tk.Entry(root)
#entry_K.pack()

correct, n, l, k = False, 0, 0, 0


def create_nlk_window():
    global correct, N, L, K, coords
    N = (n_entry.get())
    L = (l_entry.get())
    K = (k_entry.get())
    # Закрытие главного окна
    root.destroy()

    if not N.isdigit() or not K.isdigit() or not L.isdigit():
        wrong_root = tk.Tk()
        wrong_root.title('ОШИБКА!')
        wrong_root.geometry('200x50')
        tk.Label(master=wrong_root, text='Неверный формат введенных данных').pack()

        def close():
            wrong_root.destroy()

        tk.Button(master=wrong_root, text='Продолжить', comand=close()).pack()
        wrong_root.mainloop()
    else:
        N, L, K = int(N), int(L), int(K)
        correct = True


enter_button = tk.Button(text='Ввод', command=create_nlk_window)
enter_button.pack()
root.mainloop()

if not correct:
    raise TypeError('Тип введенных данных неверен')


def create_k_window(entry_lines: List):
    for i in entry_lines:
        coords_x_y = (tuple(i.get().split()))
        if len(coords_x_y) == 2 and False not in tuple(map(lambda x: x.isdigit(), coords_x_y)):
            coords.append(int(coords_x_y[0], int(coords_x_y[1])))


coords = []
kw = tk.Tk()
enter_base = [tk.Entry(master=kw) for j in range(K)]
for h in range(K):
    enter_base[h].pack()

enter_button = (tk.Button(text='Ввод координат x и y', comand=create_k_window(enter_base)))
enter_button.pack()
kw.mainloop()


class Square:
    def __init__(self):
        pass

    @staticmethod
    def draw_chessboard():
        square_size = WIDTH // N

        for i in range(N):
            for j in range(N):
                if (i + j) % 2 == 0:
                    pygame.draw.rect(screen, WHITE, (i * square_size, j * square_size, square_size, square_size))
                else:
                    pygame.draw.rect(screen, BLACK, (i * square_size, j * square_size, square_size, square_size))
# Список для хранения координат фигур


def l_alhoritm():
    global correct, N, L, K, coords, solution
    N = (n_entry.get())
    L = (l_entry.get())
    K = (k_entry.get())

    def get_existing_coords(coords: List) -> List:
        """
        Функция для поиска существующих координат
        '0 1', '2 5', '4 3' - [(0, 1), (2, 5), (4, 3)]
        """
        all_coords = []
        for cords in coords[1:]:
            coords_x_y = [int(el) for el in cords.split(' ')]  # Разделение координат x y на подстроки
            all_coords.append((coords_x_y[0], coords_x_y[1]))  # добавление координат в соответствии с их расположением
        return all_coords

    def generate_free_coords(existing_coords: List, N: int) -> List:
        """
        Генератор свободных координат
        """
        free_coords = []
        for x in range(N):
            for y in range(N):
                if (x, y) not in existing_coords:  # Проверка на наличие свободн. координат в сущетв.
                    free_coords.append((x, y))
        return free_coords

    def generate_impossible_coords(existing_coords: List, N: int) -> List:
        """
        Генерация невозможных координат
        """
        impossible_coords = []
        for x, y in existing_coords:
            impossible_coords.append((x, y + 1))
            impossible_coords.append((x, y - 1))
            impossible_coords.append((x + 1, y))
            impossible_coords.append((x - 1, y))
            impossible_coords.append((x + 1, y + 1))
            impossible_coords.append((x + 2, y + 2))
            impossible_coords.append((x + 3, y + 3))
            impossible_coords.append((x - 1, y - 1))
            impossible_coords.append((x - 2, y - 2))
            impossible_coords.append((x - 3, y - 3))
            impossible_coords.append((x - 1, y + 1))
            impossible_coords.append((x - 2, y + 2))
            impossible_coords.append((x - 3, y + 3))
            impossible_coords.append((x + 1, y - 1))
            impossible_coords.append((x + 2, y - 2))
            impossible_coords.append((x + 3, y - 3))
        return [el for el in impossible_coords if el[0] in range(N) and el[1] in range(N)]

    def generate_possible_coords(free_coords: List, impossible_coords: List):
        """
        Генерация возможных координат
        """
        return [cords for cords in free_coords if cords not in impossible_coords]  # Возврат координат с проверкой

    def is_double(variant_1: List, coord_to_add: tuple) -> bool:
        """
        Проверка на дубликаты
        """
        for coord in variant_1:
            if coord == coord_to_add:
                return True
        return False

    def move(result: List, existing_coords: List, N: int, free_coords: List) -> List:
        """
        Функция ходов для 2ой и более фигур, которые нужно расставить
        """
        new_result = []
        for cords in result:
            impossible_coords = generate_impossible_coords(existing_coords + cords, N)
            possible_coords = generate_possible_coords(free_coords, impossible_coords)
            for possible_coord in possible_coords:
                if is_double(cords, possible_coord) == False:
                    new_result.append(cords + [possible_coord])  # добавление нового результата
        return new_result


    def get_solution(coords_list: list, N: int, L: int) -> List:
        """
        Функция на получение решений
        """
        existing_coords = get_existing_coords(coords_list)
        if L == 0:
            return [existing_coords]

        free_coords = generate_free_coords(existing_coords, N)
        impossible_coords = generate_impossible_coords(existing_coords, N)
        possible_coords = generate_possible_coords(free_coords, impossible_coords)
        result = [[variant] for variant in possible_coords]
        for i in range(L - 1):
            result = move(result, existing_coords, N, free_coords)  # Решение учитывая нужные параметры
        return [existing_coords + i for i in result]

    #solution = get_solution(coords, N, L)
    #solution = list(map(lambda x: x.sort(), solution))
    #solution = list(map(lambda x: list(x), solution))
    #solution.sort()


def draw_k_pieces():
    square_size = WIDTH // N
    for piece in coords:
        x, y = piece
        pygame.draw.circle(screen, RED, (x * square_size + square_size // 2, y * square_size + square_size // 2),
                           square_size // 2)
        pygame.draw.circle(screen, BLUE, (
            (x * square_size + square_size // 2) + square_size, (y * square_size + square_size // 2) + square_size),
                           square_size // 2)
        pygame.draw.circle(screen, BLUE, (
            (x * square_size + square_size // 2) - square_size, (y * square_size + square_size // 2) - square_size),
                           square_size // 2)
        pygame.draw.circle(screen, BLUE, (
            (x * square_size + square_size // 2) + 2 * square_size,
            (y * square_size + square_size // 2) + 2 * square_size),
                           square_size // 2)
        pygame.draw.circle(screen, BLUE, (
            (x * square_size + square_size // 2) - 2 * square_size,
            (y * square_size + square_size // 2) - 2 * square_size),
                           square_size // 2)
        pygame.draw.circle(screen, BLUE, (
            (x * square_size + square_size // 2) + 3 * square_size,
            (y * square_size + square_size // 2) + 3 * square_size),
                           square_size // 2)
        pygame.draw.circle(screen, BLUE, (
            (x * square_size + square_size // 2) - 3 * square_size,
            (y * square_size + square_size // 2) - 3 * square_size),
                           square_size // 2)
        pygame.draw.circle(screen, BLUE, (
            ((x + 2) * square_size + square_size // 2) - square_size,
            (y * square_size + square_size // 2) - square_size),
                           square_size // 2)
        pygame.draw.circle(screen, BLUE, (
            ((x + 3) * square_size + square_size // 2) - square_size,
            ((y - 1) * square_size + square_size // 2) - square_size),
                           square_size // 2)
        pygame.draw.circle(screen, BLUE, (
            ((x + 4) * square_size + square_size // 2) - square_size,
            ((y - 2) * square_size + square_size // 2) - square_size),
                           square_size // 2)
        pygame.draw.circle(screen, BLUE, (
            (x * square_size + square_size // 2) - square_size,
            ((y + 2) * square_size + square_size // 2) - square_size),
                           square_size // 2)
        pygame.draw.circle(screen, BLUE, (
            ((x - 1) * square_size + square_size // 2) - square_size,
            ((y + 3) * square_size + square_size // 2) - square_size),
                           square_size // 2)
        pygame.draw.circle(screen, BLUE, (
            ((x - 2) * square_size + square_size // 2) - square_size,
            ((y + 4) * square_size + square_size // 2) - square_size),
                           square_size // 2)
        pygame.draw.circle(screen, BLUE, (
            ((x + 1) * square_size + square_size // 2) - square_size,
            (y * square_size + square_size // 2) - square_size),
                           square_size // 2)
        pygame.draw.circle(screen, BLUE, (
            ((x + 1) * square_size + square_size // 2) - square_size,
            ((y + 2) * square_size + square_size // 2) - square_size),
                           square_size // 2)
        pygame.draw.circle(screen, BLUE, (
            (x * square_size + square_size // 2) - square_size,
            ((y + 1) * square_size + square_size // 2) - square_size),
                           square_size // 2)
        pygame.draw.circle(screen, BLUE, (
            ((x + 2) * square_size + square_size // 2) - square_size,
            ((y + 1) * square_size + square_size // 2) - square_size),
                           square_size // 2)


def draw_l_pieces():
    square_size = WIDTH // N
    for piece in solution:
        x, y = piece
        pygame.draw.circle(screen, GREEN,
                           (x * square_size + square_size // 2, y * square_size + square_size // 2),
                           square_size // 2)
        pygame.draw.circle(screen, YELLOW, (
            (x * square_size + square_size // 2) + square_size,
            (y * square_size + square_size // 2) + square_size),
                           square_size // 2)
        pygame.draw.circle(screen, YELLOW, (
            (x * square_size + square_size // 2) - square_size,
            (y * square_size + square_size // 2) - square_size),
                           square_size // 2)
        pygame.draw.circle(screen, YELLOW, (
            (x * square_size + square_size // 2) + 2 * square_size,
            (y * square_size + square_size // 2) + 2 * square_size),
                           square_size // 2)
        pygame.draw.circle(screen, YELLOW, (
            (x * square_size + square_size // 2) - 2 * square_size,
            (y * square_size + square_size // 2) - 2 * square_size),
                           square_size // 2)
        pygame.draw.circle(screen, YELLOW, (
            (x * square_size + square_size // 2) + 3 * square_size,
            (y * square_size + square_size // 2) + 3 * square_size),
                           square_size // 2)
        pygame.draw.circle(screen, YELLOW, (
            (x * square_size + square_size // 2) - 3 * square_size,
            (y * square_size + square_size // 2) - 3 * square_size),
                           square_size // 2)
        pygame.draw.circle(screen, YELLOW, (
            ((x + 2) * square_size + square_size // 2) - square_size,
            (y * square_size + square_size // 2) - square_size),
                           square_size // 2)
        pygame.draw.circle(screen, YELLOW, (
            ((x + 3) * square_size + square_size // 2) - square_size,
            ((y - 1) * square_size + square_size // 2) - square_size),
                           square_size // 2)
        pygame.draw.circle(screen, YELLOW, (
            ((x + 4) * square_size + square_size // 2) - square_size,
            ((y - 2) * square_size + square_size // 2) - square_size),
                           square_size // 2)
        pygame.draw.circle(screen, YELLOW, (
            (x * square_size + square_size // 2) - square_size,
            ((y + 2) * square_size + square_size // 2) - square_size),
                           square_size // 2)
        pygame.draw.circle(screen, YELLOW, (
            ((x - 1) * square_size + square_size // 2) - square_size,
            ((y + 3) * square_size + square_size // 2) - square_size),
                           square_size // 2)
        pygame.draw.circle(screen, YELLOW, (
            ((x - 2) * square_size + square_size // 2) - square_size,
            ((y + 4) * square_size + square_size // 2) - square_size),
                           square_size // 2)
        pygame.draw.circle(screen, YELLOW, (
            ((x + 1) * square_size + square_size // 2) - square_size,
            (y * square_size + square_size // 2) - square_size),
                           square_size // 2)
        pygame.draw.circle(screen, YELLOW, (
            ((x + 1) * square_size + square_size // 2) - square_size,
            ((y + 2) * square_size + square_size // 2) - square_size),
                           square_size // 2)
        pygame.draw.circle(screen, YELLOW, (
            (x * square_size + square_size // 2) - square_size,
            ((y + 1) * square_size + square_size // 2) - square_size),
                           square_size // 2)
        pygame.draw.circle(screen, YELLOW, (
            ((x + 2) * square_size + square_size // 2) - square_size,
            ((y + 1) * square_size + square_size // 2) - square_size),
                           square_size // 2)

# Создание нового окна Tkinter

new_window = tk.Toplevel()
new_window.title("Шахматная доска")

# Создание холста для Pygame
frame = tk.Frame(new_window, width=WIDTH, height=HEIGHT)
frame.pack()

# Инициализация Pygame на холсте
os.environ['SDL_WINDOWID'] = str(frame.winfo_id())
pygame.display.init()
pygame.display.set_mode((WIDTH, HEIGHT))

# Функция для отображения шахматной доски и фигур


def draw_canvas():
    Square.draw_chessboard()
    draw_k_pieces()
    draw_l_pieces()
    pygame.display.flip()

# Отображение шахматной доски и фигур

draw_canvas()

# Функция для сохранения решения в файл


def save_solution_to_file():
    if coords != 0:
        with open("output.txt", 'a') as file:
            for variant in coords:
                file.write(str(variant)[1:-1] + '\n')
        messagebox.showinfo("Решение сохранено", "Решение сохранено в файл 'output.txt'.")
    else:
        with open("output.txt", 'a') as file:
            file.write('No solution')
        messagebox.showinfo("Решение не найдено")
    new_window.destroy()

# Функция для обработки закрытия окна


def handle_window_close():
    if len(coords) == K:
        response = messagebox.askquestion("Добавить решение?",
                                          "Решение найдено. Хотите добавить его в файл 'output.txt'?")
        if response == "yes":
            save_solution_to_file()
    else:
        with open("output.txt", 'a') as file:
            file.write("no solution\n")
        messagebox.showinfo("Решение не найдено",
                            "Решений не найдено. Записано 'no solution' в файл 'output.txt'.")
        new_window.destroy()

# Привязка функции handle_window_close() к событию закрытия окна


new_window.protocol("WM_DELETE_WINDOW", handle_window_close)

# Обновление окна при изменении размера


def on_resize(event, width, height):
    pygame.display.set_mode((width, height))
    draw_canvas()


new_window.bind("", lambda event: on_resize(event, event.width, event.height))

# Создание кнопки для сохранения решения в файл
save_button = tk.Button(new_window, text="Добавить решение", command=save_solution_to_file)
save_button.pack()

# Запуск главного цикла Tkinter
new_window.mainloop()